# Build Final Priority Index with Social Vulnerability

This notebook adds social vulnerability indicators to the municipality-level recovery access gap index.

The goal is to create a final priority score that combines:

1. overdose and EMS burden
2. recovery service access
3. social vulnerability

The output will support the visual dashboard and final project narrative.

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
processed_dir = DATA_DIR / "processed"
raw_dir = DATA_DIR / "raw"

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Processed folder:", processed_dir)
print("Raw folder:", raw_dir)
print("Processed exists:", processed_dir.exists())
print("Raw exists:", raw_dir.exists())

In [ ]:
print("Processed files:")
for path in sorted(processed_dir.glob("*")):
    print("-", path.name)

print("\nRaw files:")
for path in sorted(raw_dir.glob("*")):
    print("-", path.name)

## Load final gap index and tract-level SVI data

This section loads the municipality-level recovery access gap index and the cleaned tract-level Social Vulnerability Index dataset.

In [ ]:
gap = pd.read_csv(processed_dir / "municipality_recovery_access_gap_index.csv")
svi = pd.read_csv(processed_dir / "svi_tract_clean.csv")

print("Gap rows:", len(gap))
print("SVI rows:", len(svi))

print("\nGap columns:")
print(gap.columns.tolist())

print("\nSVI columns:")
print(svi.columns.tolist())

In [ ]:
svi.head()

In [ ]:
svi_raw_dir = raw_dir / "svi"

print("SVI raw folder exists:", svi_raw_dir.exists())

for path in sorted(svi_raw_dir.rglob("*")):
    print(path.relative_to(svi_raw_dir))

In [ ]:
tract_shapes_dir = raw_dir / "svi" / "tract_shapes"

for path in sorted(tract_shapes_dir.glob("*")):
    print(path.name)

## Join tract-level SVI to municipality boundaries

This section loads Massachusetts Census tract boundaries and prepares them for spatial aggregation to municipalities.

In [ ]:
tract_shapes_path = raw_dir / "svi" / "tract_shapes" / "tl_2022_25_tract.shp"

tracts_geo = gpd.read_file(tract_shapes_path)

print("Tract shape rows:", len(tracts_geo))
print("Tract shape CRS:", tracts_geo.crs)
print("Tract shape columns:")
print(tracts_geo.columns.tolist())

tracts_geo.head()

In [ ]:
# Make sure FIPS/GEOID keys are strings
svi["FIPS"] = svi["FIPS"].astype(str).str.zfill(11)
tracts_geo["GEOID"] = tracts_geo["GEOID"].astype(str).str.zfill(11)

svi_geo = tracts_geo.merge(
    svi,
    left_on="GEOID",
    right_on="FIPS",
    how="left"
)

print("SVI tract GeoDataFrame rows:", len(svi_geo))
print("Missing overall SVI:", svi_geo["RPL_THEMES"].isna().sum())

svi_geo[[
    "GEOID",
    "FIPS",
    "E_TOTPOP",
    "RPL_THEMES",
    "RPL_THEME1",
    "RPL_THEME2",
    "RPL_THEME3",
    "RPL_THEME4",
]].head()

In [ ]:
towns_geo = gpd.read_file(processed_dir / "ma_municipalities_clean.geojson")

print("Municipality rows:", len(towns_geo))
print("Municipality CRS:", towns_geo.crs)
print("Municipality columns:")
print(towns_geo.columns.tolist())

towns_geo.head()

In [ ]:
# Use a projected CRS for accurate area calculations in Massachusetts
target_crs = "EPSG:26986"  # NAD83 / Massachusetts Mainland meters

towns_proj = towns_geo.to_crs(target_crs)
svi_proj = svi_geo.to_crs(target_crs)

print("Towns CRS:", towns_proj.crs)
print("SVI CRS:", svi_proj.crs)

In [ ]:
# Keep only needed town and SVI columns before overlay
towns_for_overlay = towns_proj[[
    "TOWN",
    "TOWN_ID",
    "COUNTY",
    "geometry"
]].copy()

svi_for_overlay = svi_proj[[
    "GEOID",
    "E_TOTPOP",
    "RPL_THEMES",
    "RPL_THEME1",
    "RPL_THEME2",
    "RPL_THEME3",
    "RPL_THEME4",
    "EP_NOVEH",
    "EPL_NOVEH",
    "EP_POV150",
    "EPL_POV150",
    "EP_UNEMP",
    "EPL_UNEMP",
    "EP_NOHSDP",
    "EPL_NOHSDP",
    "geometry"
]].copy()

# Drop tracts without overall SVI
svi_for_overlay = svi_for_overlay.dropna(subset=["RPL_THEMES"])

tract_town_overlay = gpd.overlay(
    svi_for_overlay,
    towns_for_overlay,
    how="intersection"
)

print("Overlay rows:", len(tract_town_overlay))
print("Columns:")
print(tract_town_overlay.columns.tolist())

tract_town_overlay.head()

In [ ]:
# Area of each intersected tract-town piece
tract_town_overlay["overlap_area"] = tract_town_overlay.geometry.area

# Total area of each original tract across all town intersections
tract_area_totals = (
    tract_town_overlay
    .groupby("GEOID", as_index=False)["overlap_area"]
    .sum()
    .rename(columns={"overlap_area": "tract_overlay_total_area"})
)

tract_town_overlay = tract_town_overlay.merge(
    tract_area_totals,
    on="GEOID",
    how="left"
)

# Share of each tract that falls inside each town
tract_town_overlay["area_weight"] = (
    tract_town_overlay["overlap_area"] /
    tract_town_overlay["tract_overlay_total_area"]
)

print("Overlay rows:", len(tract_town_overlay))
print("Area weight min:", tract_town_overlay["area_weight"].min())
print("Area weight max:", tract_town_overlay["area_weight"].max())

tract_town_overlay[[
    "GEOID",
    "TOWN",
    "COUNTY",
    "E_TOTPOP",
    "RPL_THEMES",
    "area_weight",
]].head()

In [ ]:
svi_metric_cols = [
    "RPL_THEMES",
    "RPL_THEME1",
    "RPL_THEME2",
    "RPL_THEME3",
    "RPL_THEME4",
    "EP_NOVEH",
    "EPL_NOVEH",
    "EP_POV150",
    "EPL_POV150",
    "EP_UNEMP",
    "EPL_UNEMP",
    "EP_NOHSDP",
    "EPL_NOHSDP",
]

# Weight population and SVI metrics by tract-town area overlap
tract_town_overlay["weighted_population"] = (
    tract_town_overlay["E_TOTPOP"] * tract_town_overlay["area_weight"]
)

for col in svi_metric_cols:
    tract_town_overlay[f"weighted_{col}"] = (
        tract_town_overlay[col] * tract_town_overlay["weighted_population"]
    )

weighted_cols = [f"weighted_{col}" for col in svi_metric_cols]

town_svi = (
    tract_town_overlay
    .groupby(["TOWN", "TOWN_ID", "COUNTY"], as_index=False)
    .agg(
        estimated_population=("weighted_population", "sum"),
        **{col: (col, "sum") for col in weighted_cols}
    )
)

# Convert weighted sums back to population-weighted averages
for col in svi_metric_cols:
    town_svi[col] = town_svi[f"weighted_{col}"] / town_svi["estimated_population"]

town_svi = town_svi.drop(columns=weighted_cols)

town_svi["town_join"] = town_svi["TOWN"].astype(str).str.upper().str.strip()

print("Town SVI rows:", len(town_svi))
print("Missing overall SVI:", town_svi["RPL_THEMES"].isna().sum())

town_svi.sort_values("RPL_THEMES", ascending=False).head(20)

In [ ]:
town_svi_path = processed_dir / "municipality_svi_table.csv"

town_svi.to_csv(town_svi_path, index=False)

print("Saved:", town_svi_path)
print("File exists:", town_svi_path.exists())
print("Rows:", len(town_svi))
print("Columns:", len(town_svi.columns))

## Build final priority index

This section joins municipality-level SVI indicators to the recovery access gap index and creates a final priority score.

In [ ]:
final_priority = gap.merge(
    town_svi[[
        "town_join",
        "estimated_population",
        "RPL_THEMES",
        "RPL_THEME1",
        "RPL_THEME2",
        "RPL_THEME3",
        "RPL_THEME4",
        "EP_NOVEH",
        "EPL_NOVEH",
        "EP_POV150",
        "EPL_POV150",
        "EP_UNEMP",
        "EPL_UNEMP",
        "EP_NOHSDP",
        "EPL_NOHSDP",
    ]],
    on="town_join",
    how="left"
)

print("Final priority rows:", len(final_priority))
print("Expected municipalities:", len(gap))
print("Missing SVI:", final_priority["RPL_THEMES"].isna().sum())

final_priority.head()

In [ ]:
# Rename the main SVI percentile for readability
final_priority["social_vulnerability_pct"] = final_priority["RPL_THEMES"]

# Higher score = high recovery access gap + high social vulnerability
final_priority["final_priority_score"] = (
    final_priority["recovery_access_gap_score"] *
    (1 + final_priority["social_vulnerability_pct"])
)

final_priority.sort_values(
    "final_priority_score",
    ascending=False
)[[
    "TOWN",
    "COUNTY",
    "estimated_population",
    "recovery_access_gap_score",
    "social_vulnerability_pct",
    "final_priority_score",
    "gap_category",
    "avg_deaths_2021_2023",
    "avg_ems_incidents_2022_2023",
    "service_diversity_score",
]].head(25)

In [ ]:
def assign_priority_category(score):
    if score >= 1.25:
        return "Very high priority"
    elif score >= 0.90:
        return "High priority"
    elif score >= 0.50:
        return "Moderate priority"
    else:
        return "Lower priority"


final_priority["priority_category"] = final_priority["final_priority_score"].apply(
    assign_priority_category
)

final_priority["priority_category"].value_counts()

In [ ]:
final_priority.sort_values(
    "final_priority_score",
    ascending=False
)[[
    "TOWN",
    "COUNTY",
    "estimated_population",
    "avg_deaths_2021_2023",
    "avg_ems_incidents_2022_2023",
    "service_diversity_score",
    "recovery_access_gap_score",
    "social_vulnerability_pct",
    "final_priority_score",
    "gap_category",
    "priority_category",
]].head(25)

In [ ]:
final_priority_csv_path = processed_dir / "municipality_final_priority_index.csv"

final_priority.to_csv(final_priority_csv_path, index=False)

print("Saved:", final_priority_csv_path)
print("File exists:", final_priority_csv_path.exists())
print("Rows:", len(final_priority))
print("Columns:", len(final_priority.columns))

## Create dashboard-ready GeoJSON with final priority score

This section joins the final priority index back to municipality boundaries for the interactive dashboard map.

In [ ]:
towns_geo = gpd.read_file(processed_dir / "ma_municipalities_clean.geojson")

towns_geo["town_join"] = towns_geo["TOWN"].astype(str).str.upper().str.strip()

final_priority_geo = towns_geo.merge(
    final_priority,
    on="town_join",
    how="left",
    suffixes=("_geo", "")
)

print("Rows:", len(final_priority_geo))
print("Expected municipalities:", len(towns_geo))
print("Missing final priority scores:", final_priority_geo["final_priority_score"].isna().sum())

final_priority_geo[[
    "TOWN_geo",
    "COUNTY_geo",
    "recovery_access_gap_score",
    "social_vulnerability_pct",
    "final_priority_score",
    "priority_category",
]].head()

In [ ]:
final_priority_geo_clean = final_priority_geo.copy()

# Keep clean municipality and county names
final_priority_geo_clean["TOWN"] = final_priority_geo_clean["TOWN_geo"]
final_priority_geo_clean["COUNTY"] = final_priority_geo_clean["COUNTY_geo"]

# Drop duplicate merge columns
cols_to_drop = [
    "TOWN_geo",
    "COUNTY_geo",
    "TOWN_ID_geo",
]

final_priority_geo_clean = final_priority_geo_clean.drop(
    columns=[col for col in cols_to_drop if col in final_priority_geo_clean.columns]
)

final_priority_geojson_path = processed_dir / "municipality_final_priority_index.geojson"

final_priority_geo_clean.to_file(final_priority_geojson_path, driver="GeoJSON")

print("Saved:", final_priority_geojson_path)
print("File exists:", final_priority_geojson_path.exists())
print("Rows:", len(final_priority_geo_clean))

In [ ]:
print("Final priority CSV exists:", final_priority_csv_path.exists())
print("Final priority GeoJSON exists:", final_priority_geojson_path.exists())

print("\nPriority categories:")
print(final_priority["priority_category"].value_counts())

print("\nTop 10 final priority municipalities:")
final_priority.sort_values(
    "final_priority_score",
    ascending=False
)[[
    "TOWN",
    "COUNTY",
    "estimated_population",
    "recovery_access_gap_score",
    "social_vulnerability_pct",
    "final_priority_score",
    "priority_category",
]].head(10)